In [ ]:
df_train = pd.read_csv("path/to/train.csv")
df_test = pd.read_csv("path/to/test.csv")

## 1. Separating Features from the Target

We define the target variable and the columns that must **not** be used by the model:
- **Identifiers** (`ID`, `customer_id`): they have no predictive power and can lead to overfitting.
- **`purchase_date`:** dropped in this baseline version. It can be reused later through derived features (month, day of week, etc.).
- **Target:** excluded from the features to avoid *data leakage*.

The remaining columns become the features (`X`) and the target (`y`). We also keep the test `ID`s to build the submission file, and we ensure that `X_test` has **exactly the same columns, in the same order,** as `X`.

In [ ]:
target_col = "repeat_purchase_90d"
ignore_cols = ["ID", "customer_id", "purchase_date", target_col]

# Seleção automática de features preditivas (apenas numéricas e categóricas úteis)
feature_cols = [c for c in df_train.columns if c not in ignore_cols]

X = df_train[feature_cols].copy()
y = df_train[target_col].copy()

# Guardar os IDs do ficheiro de teste para o ficheiro final de submissão
test_ids = df_test_raw["ID"].copy()

# APLICAR NO TEST.CSV AS MESMAS TRANSFORMAÇÕES/FEATURES FEITAS NO TRAIN
# (Garantir que X_test tem exatamente as mesmas colunas que X)
X_test = df_test_raw[
    [c for c in feature_cols if c in df_test_raw.columns]
].copy()

## 2. Training with Stratified K-Fold (Cross-Validation)

To evaluate the model reliably, we use **5-fold stratified cross-validation**. Stratification preserves the class proportions (repeat vs. non-repeat buyers) in every fold, which is important when the target is imbalanced.

In each fold:
1. We train the model on the fold's training data.
2. We predict probabilities on the validation set and store them in `oof_preds` (*out-of-fold*), so every customer is predicted by a model that has never seen them.
3. We compute the **ROC-AUC** (quality of the probability ranking) and the **F1-Score** at a 0.5 threshold.
4. We add the fold's test predictions, divided by 5, which gives the **average probability** across the 5 models.

We use `HistGradientBoostingClassifier` as the baseline model: it is fast, robust, and natively handles missing values (NaN).

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(df_train))
test_preds = np.zeros(len(df_test_raw))

# Modelo baseline simples e robusto (LightGBM/HistGradientBoosting lida bem com NaNs e categóricas)
model = HistGradientBoostingClassifier(random_state=42)

auc_scores, f1_scores = [], []

print("=== INÍCIO DA CROSS-VALIDATION ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
  X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
  X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

  # Treinar o modelo
  model.fit(X_tr, y_tr)

  # Prever probabilidades no conjunto de validação
  val_probs = model.predict_proba(X_va)[:, 1]
  oof_preds[val_idx] = val_probs

  # Avaliar o fold
  fold_auc = roc_auc_score(y_va, val_probs)
  fold_f1 = f1_score(y_va, (val_probs >= 0.5).astype(int))

  auc_scores.append(fold_auc)
  f1_scores.append(fold_f1)

  print(
      f"Fold {fold+1} | ROC-AUC: {fold_auc:.4f} | F1-Score (threshold=0.5):"
      f" {fold_f1:.4f}"
  )

  # Somar previsões acumuladas para o dataset de teste
  test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

## 3. Overall Performance Evaluation (Out-of-Fold)

After running all 5 folds, we evaluate the model using **all** out-of-fold predictions together (`oof_preds`). Each customer was predicted by a model that did not see them during training, which gives a realistic estimate that is more stable than the average of individual folds.

Metrics used:
- **ROC-AUC (mean across folds):** average of the AUC computed in each fold. It also shows the variability between folds.
- **ROC-AUC (overall OOF):** measures the model's ability to rank customers who will buy again above those who will not, regardless of the threshold.
- **Log-Loss (overall OOF):** evaluates the quality of the predicted **probabilities**, heavily penalizing confident but wrong predictions. Lower is better.

These values serve as a *baseline* to compare future improvements (new features, other models, hyperparameter tuning).

In [ ]:
overall_auc = roc_auc_score(y, oof_preds)
overall_logloss = log_loss(y, oof_preds)

print("\n=== DESEMPENHO GLOBAL DO MODELO BASELINE ===")
print(f"• Mean ROC-AUC (Cross-Validation): {np.mean(auc_scores):.4f}")
print(f"• Overall Out-of-Fold ROC-AUC:    {overall_auc:.4f}")
print(f"• Overall Out-of-Fold Log-Loss:   {overall_logloss:.4f}")


## 4. Submission File Generation

We build the submission file in the `sample_submission.csv` format, with two columns:
- `ID`: identifier of each record in the test set.
- `repeat_purchase_90d`: **probability** that the customer will buy again within the next 90 days, computed as the average of the predictions from the 5 models trained during cross-validation.

The result is saved as a CSV (without the index), and we display the first rows to confirm the format is correct.

In [ ]:
submission = pd.DataFrame(
    {"ID": test_ids, "repeat_purchase_90d": test_preds}
)

submission_file = "/Users/renata/Library/CloudStorage/OneDrive-NOVAIMS/mestrado/1semestre/AprendizagemAutomatica(S1)/ProjectML/ML_Grupo18/data/submission_baseline.csv"
submission.to_csv(submission_file, index=False)

print(
    f"\nFicheiro de submissão gerado com sucesso em: {submission_file}"
)
print("Amostra das previsões:")
print(submission.head())